# 1. Setup & Parameters

In [0]:
from pyspark.sql.functions import *

batch_id = dbutils.widgets.get("batch_id")

print(f"[INFO] Starting ingestion with batch_id: {batch_id}")

# 2. Define Entities

In [0]:
entities = [
    "users",
    "subscriptions",
    "subscription_changes",
    "payments",
    "support_tickets"
]

print(f"[INFO] Entities to process: {entities}")

# 3. Ingestion Loop

In [0]:
streams = []

for entity in entities:

    print(f"\n[INFO] ===== Processing entity: {entity} =====")

    raw_path = f"/Volumes/source_catalog/source_schema/raw_files/{entity}/"
    target_path = f"/Volumes/datalake_catalog/datalake_schema/landing/{entity}/"
    checkpoint_path = f"/Volumes/datalake_catalog/datalake_schema/landing/_checkpoints/{entity}/"
    schema_path = f"/Volumes/datalake_catalog/datalake_schema/landing/_schemas/{entity}/"

    print(f"[INFO] Raw path: {raw_path}")
    print(f"[INFO] Target path: {target_path}")
    print(f"[INFO] Checkpoint path: {checkpoint_path}")
    print(f"[INFO] Schema path: {schema_path}")

    df = (
        spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("cloudFiles.schemaLocation", schema_path)
        .option("cloudFiles.inferColumnTypes", "true")
        .option("cloudFiles.schemaEvolutionMode", "addNewColumns")
        .option("cloudFiles.maxFilesPerTrigger", 500)

        # 🔥 reprocess updated files
        .option("cloudFiles.allowOverwrites", "true")
        .option("cloudFiles.includeExistingFiles", "true")

        .load(raw_path)
    )

    print(f"[INFO] Schema for {entity}:")
    df.printSchema()

    df = (
        df.withColumn("dw_ingested_at", current_timestamp())
          .withColumn("dw_source_file", col("_metadata.file_path"))
          .withColumn("dw_batch_id", lit(batch_id))
    )

    # ⚠️ basic dedup (improve with business key later)
    df = df.dropDuplicates()

    df = df.repartition(4)

    print(f"[INFO] Starting stream write for {entity}...")

    query = (
        df.writeStream
        .format("delta")
        .option("checkpointLocation", checkpoint_path)
        .trigger(availableNow=True)
        .start(target_path)
    )

    streams.append((entity, query))

# 4. Wait for Completion

In [0]:
for entity, query in streams:
    print(f"[INFO] Waiting for {entity} stream to finish...")
    query.awaitTermination()
    print(f"[SUCCESS] Finished processing {entity}")

# 5. Final Summary

In [0]:
print(f"[SUCCESS] All entities processed successfully for batch_id: {batch_id}")